# Building the streaming consumer step by step

This notebook reads files as they land in `data/stream_input/` (written by `feeder_dev.ipynb`) and computes two things: a raw row-count per micro-batch, and a windowed event-time count. Run this notebook's cells up through "start both queries" *before* running the feed loop in `feeder_dev.ipynb`, so a consumer is already watching the directory.

In [15]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType
from pyspark.sql.functions import col, to_timestamp, window

from config import (
    STREAM_INPUT_DIR,
    CHECKPOINT_DIR,
    OUTPUT_DIR,
    DEFAULT_MAX_FILES_PER_TRIGGER,
    DEFAULT_WINDOW_DURATION,
    DEFAULT_WATERMARK_DELAY,
    DEFAULT_TRIGGER_INTERVAL_SEC,
)

## Step 0 — start Spark

Boilerplate, given directly: a local Spark session. `spark.sql.shuffle.partitions` is set low (default is 200) because our data volume here is tiny — 200 shuffle partitions for a few hundred rows per micro-batch would just add overhead.

In [16]:
spark = (
    SparkSession.builder
    .appName("kt1-throughput-stream")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## Step 1 — declare the schema

**Important:** when Spark's CSV reader gets both an explicit schema *and* `header=true`, it matches columns **positionally** — it just skips the header line, it does not match by column name. So these fields must be in the exact same order the feeder writes them in: `timestamp, solving_id, question_id, user_answer, elapsed_time, user_id`.

Fill in the type for each field: `LongType()` for whole numbers, `StringType()` for text — same reasoning as the pandas `dtype` dict you used in the sampler.

In [17]:
def kt1_stream_schema() -> StructType:
    return StructType([
        StructField("timestamp", LongType(), False),     # TODO
        StructField("solving_id", LongType(), False),    # TODO
        StructField("question_id", StringType(), False),   # TODO
        StructField("user_answer", StringType(), True),    # TODO
        StructField("elapsed_time", LongType(), True),   # TODO
        StructField("user_id", StringType(), False),       # TODO
    ])


schema = kt1_stream_schema()
schema

StructType([StructField('timestamp', LongType(), False), StructField('solving_id', LongType(), False), StructField('question_id', StringType(), False), StructField('user_answer', StringType(), True), StructField('elapsed_time', LongType(), True), StructField('user_id', StringType(), False)])

## Step 2 — open the stream

`readStream` looks a lot like `read`, but it doesn't load data immediately — it just describes *where to watch* and *how*. `maxFilesPerTrigger` caps how many new files Spark picks up per micro-batch, so a burst of arrivals doesn't get swallowed in one giant batch.

Fill in the blank: which config constant controls that cap?

In [18]:
raw = (
    spark.readStream
    .format("csv")
    .schema(schema)
    .option("header", "true")
    .option("maxFilesPerTrigger", DEFAULT_MAX_FILES_PER_TRIGGER)  # TODO: which config constant?
    .load(str(STREAM_INPUT_DIR))
)

raw.isStreaming

True

## Step 3 — turn the raw timestamp into an event-time column

`timestamp` is epoch **milliseconds** (e.g. `1565096190868`), but `to_timestamp()` expects epoch **seconds**. Fill in the blank: what do you divide by to convert milliseconds to seconds?

In [19]:
def with_event_time(df):
    return df.withColumn("event_time", to_timestamp(col("timestamp") / 1000))  # TODO


raw_with_time = with_event_time(raw)
raw_with_time.printSchema()

root
 |-- timestamp: long (nullable = true)
 |-- solving_id: long (nullable = true)
 |-- question_id: string (nullable = true)
 |-- user_answer: string (nullable = true)
 |-- elapsed_time: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



## Step 4 — windowed, watermarked count

Two new ideas here:
- **`withWatermark(col, delay)`** tells Spark "once you've seen an event_time this far ahead, assume anything older than `delay` behind it will never arrive — you can stop waiting and finalize that window." It advances based on the *data's own* max event-time, not wall-clock time. Because our feeder replays data in sorted order, the watermark keeps advancing every trigger, so windows actually close and finalize during the demo — even though the underlying dates are from 2017–2019.
- **`groupBy(window(col, duration))`** buckets rows into fixed time windows (e.g. hourly) and counts them — same idea as pandas `groupby`, just bucketed by time instead of by a category column.

Fill in two blanks: which column is our event-time column, and which variable holds the window size?

In [20]:
def windowed_counts(df, window_duration, watermark_delay):
    return (
        df.withWatermark( 'event_time', watermark_delay)   # TODO: which column?
          .groupBy(window(col("event_time"), window_duration))  # TODO: which variable?
          .count()
    )


windowed = windowed_counts(raw_with_time, DEFAULT_WINDOW_DURATION, DEFAULT_WATERMARK_DELAY)

## Step 5 — per-micro-batch row count, saved to disk

`foreachBatch(fn)` calls `fn(batch_df, batch_id)` once per micro-batch, handing you that trigger's slice of data as a regular (non-streaming) DataFrame — so ordinary DataFrame methods work on it.

This callback prints the row count **and** saves it as a one-row Parquet file under `data/output/batch_counts/`. The file name contains the batch id, so re-processing the same batch overwrites instead of duplicating. `processed_at` is the wall-clock time the batch finished, which lets you measure real throughput afterwards.

In [21]:
def record_batch_count(batch_df, batch_id):
    n = batch_df.count()
    print(f"[batch {batch_id}] {n} rows")

    path = OUTPUT_DIR / "batch_counts" / f"batch_{batch_id:06d}.parquet"
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(
        {"batch_id": [batch_id], "rows": [n], "processed_at": [pd.Timestamp.now()]}
    ).to_parquet(path, index=False)

## Step 5b — windowed counts, saved to disk

Same idea for the windowed query. In `update` mode `batch_df` holds only the windows whose totals changed in this batch, with the running total in `count`. The callback flattens the `window` struct into `window_start` / `window_end`, prints **all** rows (the console sink cut output off at 20), and saves them to `data/output/window_counts/`. The result is tiny, so `toPandas()` is fine here; at real scale you would write with Spark instead.

To analyse afterwards: `pd.read_parquet(OUTPUT_DIR / "window_counts")` gives one row per (batch, window). The latest `events` value per window is its current total.

In [22]:
def record_window_counts(batch_df, batch_id):
    pdf = (
        batch_df
        .select(
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("count").alias("events"),
        )
        .orderBy("window_start")
        .toPandas()
    )
    pdf.insert(0, "batch_id", batch_id)

    print(f"--- window counts, batch {batch_id} ({len(pdf)} windows) ---")
    print(pdf.to_string(index=False))

    path = OUTPUT_DIR / "window_counts" / f"batch_{batch_id:06d}.parquet"
    path.parent.mkdir(parents=True, exist_ok=True)
    pdf.to_parquet(path, index=False)

## Step 6 — start both queries

Two separate `writeStream` queries, each with its own checkpoint: the raw count is stateless (just counts whatever arrived this trigger), but the windowed count is stateful across triggers (Spark has to remember partial window totals between micro-batches until the watermark closes them) — they can't share one query.

`.start()` doesn't block — it launches both in the background and returns immediately.

In [23]:
query_raw = (
    raw.writeStream
    .foreachBatch(record_batch_count)
    .option("checkpointLocation", str(CHECKPOINT_DIR / "raw_counts"))
    .trigger(processingTime=f"{DEFAULT_TRIGGER_INTERVAL_SEC} seconds")
    .start()
)

query_windowed = (
    windowed.writeStream
    .outputMode("update")
    .foreachBatch(record_window_counts)
    .option("checkpointLocation", str(CHECKPOINT_DIR / "windowed_counts"))
    .trigger(processingTime=f"{DEFAULT_TRIGGER_INTERVAL_SEC} seconds")
    .start()
)

print("Streaming started. Now go run the feed loop in feeder_dev.ipynb, then run the next cell here to watch output.")

26/09/21 18:41:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/09/21 18:41:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming started. Now go run the feed loop in feeder_dev.ipynb, then run the next cell here to watch output.


## Step 7 — watch it run

This blocks and prints output live as micro-batches arrive. Use the notebook's interrupt/stop button to break out whenever you're done watching — it won't corrupt anything, the queries keep running in the background until you explicitly stop them in the next cell.

In [ ]:
spark.streams.resetTerminated()
print([q.name or q.id for q in spark.streams.active])   # should list your 2 live queries
spark.streams.awaitAnyTermination()


['0c6afb82-5682-402c-8a93-1396c4315b33', 'f7ab04ff-fdba-428f-bb18-3be18eb5ac47']
[batch 0] 200 rows
[batch 1] 200 rows
--- window counts, batch 0 (12 windows) ---
 batch_id        window_start          window_end  events
        0 2017-07-01 13:00:00 2017-07-01 14:00:00       3
        0 2017-07-01 14:00:00 2017-07-01 15:00:00       9
        0 2017-07-02 08:00:00 2017-07-02 09:00:00      20
        0 2017-07-02 09:00:00 2017-07-02 10:00:00      15
        0 2017-07-02 10:00:00 2017-07-02 11:00:00      12
        0 2017-07-02 18:00:00 2017-07-02 19:00:00       6
        0 2017-07-07 14:00:00 2017-07-07 15:00:00      35
        0 2017-07-11 16:00:00 2017-07-11 17:00:00      30
        0 2017-07-12 05:00:00 2017-07-12 06:00:00      30
        0 2017-07-17 20:00:00 2017-07-17 21:00:00      35
        0 2017-07-17 21:00:00 2017-07-17 22:00:00       4
        0 2017-07-19 15:00:00 2017-07-19 16:00:00       1
[batch 2] 200 rows
--- window counts, batch 1 (8 windows) ---
 batch_id        wind

## Cleanup — stop both queries when you're done

In [ ]:
query_raw.stop()
query_windowed.stop()
print("stopped")